In [26]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from src.utils import read_obs

In [35]:
obs = read_obs("/root/capsule/data/filtered_adata/CaH_final_nuclei.2025-12-29.h5ad")
cell_bool = obs.groupby("Donor ID")["Supertype"].value_counts().unstack("Supertype").apply(lambda x: np.any(x < 10))
cells_to_remove= cell_bool[cell_bool].index.str.replace(" ", "_") 

/tmp/ipykernel_70865/3051182257.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cell_bool = obs.groupby("Donor ID")["Supertype"].value_counts().unstack("Supertype").apply(lambda x: np.any(x < 10))


In [36]:
cells_to_remove

Index(['Astro_4', 'Astro_5', 'Astro_6-SEAAD', 'BG_LAMP5-CXCL14_GABA', 'Endo_1',
       'Endo_3', 'Ependymal', 'LSX_GABA', 'Lymphocyte', 'Micro-PVM_2_1-SEAAD',
       'Micro-PVM_4-SEAAD', 'Monocyte', 'OB-in_Frmd7_Gaba', 'OPC_1',
       'OPC_2_1-SEAAD', 'OPC_2_2-SEAAD', 'Oligo_1', 'Oligo_2_1-SEAAD',
       'Oligo_3', 'Pericyte_2-SEAAD', 'SMC-SEAAD', 'STR_PVALB-RSPO2_GABA',
       'STR_SST-RSPO2_GABA', 'STR_TAC3-PLPP4-LHX8_GABA', 'STR_VIP_GABA',
       'STRd_D1D2_Hybrid_MSN', 'STRd_D2_StrioMat_Hybrid_MSN', 'STRv_D1_MSN',
       'STRv_D1_NUDAP_MSN', 'STRv_D2_MSN', 'Sst_Chodl_1', 'Sst_Chodl_2',
       'Vip_1', 'Vip_2', 'Vip_4', 'Vip_5', 'Vip_6', 'Vip_9', 'Vip_11',
       'Vip_12', 'Vip_13', 'Vip_14', 'Vip_15', 'Vip_16', 'Vip_18', 'Vip_19',
       'Vip_21', 'Vip_23'],
      dtype='object', name='Supertype')

In [7]:
at8_df = pd.read_csv("/data/DEG_Supplemental_Tables/Supplemental Table 5.csv", index_col = 0)
sixE10_df = pd.read_csv("/data/DEG_Supplemental_Tables/Supplemental Table 6.csv", index_col = 0)

In [8]:
def mask_high_var(df, covaraiate):
    return df.apply(lambda x: x[f"logFC_{covariate}"] if x[f"se_{covariate}"] < 20 else np.nan, axis = 1)

def get_covariates(df, prefix = "se_"):
    prefix_df = df.loc[:,[col.startswith(prefix) for col in df.columns]]
    return prefix_df.columns.map(lambda x: x.removeprefix(prefix)).tolist()

In [9]:
for covariate in get_covariates(at8_df):
    at8_df[f"logFC_{covariate}"] = mask_high_var(at8_df, covariate)

In [10]:
for covariate in get_covariates(sixE10_df):
    sixE10_df[f"logFC_{covariate}"] = mask_high_var(sixE10_df, covariate)

In [14]:
at8_df["Significant"] = (at8_df['logFC_CPS_AT8'].abs() > 0.5) & (at8_df['fdr_p_CPS_AT8'] < 0.1)
sixE10_df["Significant"] = (sixE10_df['logFC_CPS_6e10'].abs() > 0.5) & (sixE10_df['fdr_p_CPS_6e10'] < 0.1)

In [15]:
at8_significant = at8_df[at8_df["Significant"]].index
sixE10_significant = sixE10_df[sixE10_df["Significant"]].index

In [16]:
both_idx = at8_significant.intersection(sixE10_significant)
at8_idx = at8_significant.difference(sixE10_significant)
sixE10_idx = sixE10_significant.difference(at8_significant)

In [19]:
at8_df

,logFC_(Intercept),logFC_Age_at_Death_binned_codes,logFC_Sex_codes,logFC_method10xV3.1_HT,logFC_APOE4_Status_codes,logFC_CPS_AT8,se_(Intercept),se_Age_at_Death_binned_codes,se_Sex_codes,se_method10xV3.1_HT,...,gene_id,var_name,var_index,fdr_p_Age_at_Death_binned_codes,fdr_p_Sex_codes,fdr_p_method10xV3.1_HT,fdr_p_APOE4_Status_codes,fdr_p_CPS_AT8,cell_type,Significant
Micro-PVM_2_1-SEAAD_AL627309.1,NaN,0.260529,-0.517282,NaN,-0.450044,3.037225,73.604779,0.673275,0.521646,1200.317575,...,1,AL627309.1,AL627309.1,0.973411,0.897785,9.999188e-01,0.972881,0.965773,Micro-PVM_2_1-SEAAD,False
Micro-PVM_2_1-SEAAD_AL627309.5,NaN,-1.494190,0.655691,NaN,-1.466823,-0.480798,56.971883,1.212134,0.956395,929.067135,...,2,AL627309.5,AL627309.5,0.881053,0.943570,9.999188e-01,0.940409,0.992645,Micro-PVM_2_1-SEAAD,False
Micro-PVM_2_1-SEAAD_LINC01409,NaN,-0.870350,0.346638,NaN,-0.755130,-0.511939,76.806784,0.497746,0.380498,1252.539610,...,3,LINC01409,LINC01409,0.859183,0.908287,9.999188e-01,0.938180,0.965773,Micro-PVM_2_1-SEAAD,False
Micro-PVM_2_1-SEAAD_LINC01128,-10.847169,0.306247,1.073078,-0.003421,-0.289869,-0.548653,0.170204,0.616174,0.477999,0.838496,...,4,LINC01128,LINC01128,0.958811,0.719170,9.999188e-01,0.989485,0.970952,Micro-PVM_2_1-SEAAD,False
Micro-PVM_2_1-SEAAD_LINC00115,NaN,-0.383347,-0.267463,NaN,-0.382461,-0.716821,59.583069,1.471836,1.161032,971.635739,...,5,LINC00115,LINC00115,0.986199,0.999873,9.999188e-01,0.999980,0.993217,Micro-PVM_2_1-SEAAD,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
STRd_D1D2_Hybrid_MSN_MAFIP,-12.286099,0.010987,-0.621993,0.702706,0.276162,-0.462454,0.104664,0.345698,0.229667,0.069419,...,24335,MAFIP,MAFIP,0.997764,0.616476,2.069956e-22,0.995286,0.866995,STRd_D1D2_Hybrid_MSN,False
STRd_D1D2_Hybrid_MSN_AC011043.1,-13.659372,-0.219846,0.739584,-0.379920,0.282818,-0.863205,0.221327,0.822834,0.540731,0.098420,...,24336,AC011043.1,AC011043.1,0.983984,0.768996,5.914875e-04,0.995286,0.907766,STRd_D1D2_Hybrid_MSN,False
STRd_D1D2_Hybrid_MSN_AL592183.1,-9.965781,0.125143,-0.021648,-0.162272,0.116245,0.295144,0.099144,0.350758,0.221124,0.022742,...,24338,AL592183.1,AL592183.1,0.978292,0.985730,1.774590e-11,0.995286,0.927066,STRd_D1D2_Hybrid_MSN,False
STRd_D1D2_Hybrid_MSN_AC240274.1,-12.778401,0.171281,-0.287994,-0.050714,0.115009,-0.125512,0.059723,0.192822,0.131870,0.086450,...,24339,AC240274.1,AC240274.1,0.919784,0.715916,6.903279e-01,0.995286,0.947868,STRd_D1D2_Hybrid_MSN,False


In [24]:
both_count = at8_df.loc[both_idx].groupby("cell_type")["var_name"].count().reset_index()
both_count['path'] = "both"

at8_count =  at8_df.loc[at8_idx].groupby("cell_type")["var_name"].count().reset_index()
at8_count['path'] = r"$CPS_{AT8}$"

sixE10_count = sixE10_df.loc[sixE10_idx].groupby("cell_type")['var_name'].count().reset_index()
sixE10_count['path'] = r"$CPS_{6E10}$"

count_df = pd.concat([both_count, at8_count, sixE10_count])

In [38]:
with plt.rc_context({"font.size": 24}):
    fig, ax = plt.subplots(figsize = (15, 10), dpi = 600,)
    sns.barplot(x = "cell_type", y = "var_name", hue_order = ["Both", "$CPS_{AT8}$", "$CPS_{6e10}$"],  palette = ['#276AB3', "#762a83", "#1b7837"], data = count_df[~count_df['cell_type'].isin(cells_to_remove)].sort_values("cell_type"), ax = ax)
    plt.xticks(rotation = 45, ha = "right")
    plt.xlabel("Cell Type")
    plt.ylabel("DEG Count")

/tmp/ipykernel_70865/2162611185.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x = "cell_type", y = "var_name", hue_order = ["Both", "$CPS_{AT8}$", "$CPS_{6e10}$"],  palette = ['#276AB3', "#762a83", "#1b7837"], data = count_df[~count_df['cell_type'].isin(cells_to_remove)].sort_values("cell_type"), ax = ax)
/tmp/ipykernel_70865/2162611185.py:3: UserWarning: 
The palette list has fewer values (3) than needed (19) and will cycle, which may produce an uninterpretable plot.
  sns.barplot(x = "cell_type", y = "var_name", hue_order = ["Both", "$CPS_{AT8}$", "$CPS_{6e10}$"],  palette = ['#276AB3', "#762a83", "#1b7837"], data = count_df[~count_df['cell_type'].isin(cells_to_remove)].sort_values("cell_type"), ax = ax)
